# [1장 통합 실습] 사내 도우미의 자료 수집 계획 세우기

세 TODO를 채우고 결과를 확인하세요.

## 만들 것

사내 Private LLM 도우미에 교육 관련 자료를 연결하려고 합니다. 직원은 교육비 지원 규정도 묻고, 지금 신청할 수 있는 자리도 묻습니다. 담당자는 자료를 한꺼번에 모으기 전에 **질문에 맞는 수집 방식과 사용 가능한 출처를 정리한 계획**이 필요합니다.

네 가지 후보를 검사해 `collection_plan.json` 하나를 만드세요. 각 후보에 수집 경로, `가능`·`주의`·`제외` 판단, 판단 근거와 확인일을 남깁니다. 실제 자료를 내려받거나 LLM을 실행하는 단계는 포함하지 않습니다.

## 제공 자료 읽기

`CANDIDATES`의 각 딕셔너리는 질문 하나와 그 질문에 사용하려는 출처입니다. 이용 조건과 주소는 모두 이 문제를 위해 만든 가상 자료입니다. `True`는 해당 조건을 확인했다는 뜻이고, `None`은 아직 모른다는 뜻입니다. 이 문제에서는 질문에 필요 없는 개인 연락처를 수집 대상에서 제외합니다.

| 후보 | 직원의 질문 | 제공된 이용 조건 |
| --- | --- | --- |
| 교육비 지원 규정 | 지원 조건과 예외가 궁금하다. | 출처를 표시하면 내부 검색용 저장을 허용한다. |
| 교육 신청 잔여석 | 지금 신청 가능한 자리가 궁금하다. | 내부 조회를 허용하며, 응답의 기준 시각을 표시해야 한다. |
| 월별 교육 신청 통계 | 최근 석 달의 변화를 비교하고 싶다. | 출처 표시 후 보관은 허용하지만 웹 경로의 robots 확인은 끝나지 않았다. |
| 교육 신청자 연락처 | 담당 부서 이름이 궁금하다. | 질문에 필요 없는 개인 전화번호가 들어 있고 재사용 조건도 불명확하다. |

`format`은 PDF·JSON·HTML·CSV와 같은 표현 형식입니다. `method`는 현재 후보가 제공되는 경로로, 파일은 `file`, API는 `api`, 웹 자동 수집은 `web`입니다. 앞으로 어떤 방식으로 사용할지 고르는 `route`와 구분하세요. 예를 들어 HTML 통계 페이지라도 과거 비교가 목적이면 시점별 스냅샷을 남기는 계획이 필요합니다.

## TODO 1 — 질문에 맞는 수집 경로 고르기

`choose_route(item)`은 후보 딕셔너리 하나를 받아 경로 이름 문자열을 반환합니다. 다음 순서대로 판단하세요. 앞에서 조건을 만족하면 뒤의 조건은 보지 않습니다.

1. `needs_history`가 참이면 `주기적 파일/스냅샷`을 반환합니다.
2. 그렇지 않고 `changes_often`이 참이면 `질문 시점 API`를 반환합니다.
3. 그렇지 않고 `needs_explanation`이 참이면 `공식 문서/RAG`를 반환합니다.
4. 세 조건이 모두 거짓이면 `정적 파일`을 반환합니다.

월별 통계에는 과거 비교와 잦은 변경이 모두 표시되어 있습니다. 이때 현재값만 고르지 않도록 순서를 지켜 주세요. 이 규칙은 주어진 질문을 비교하기 위한 출발점이며 모든 서비스에 그대로 적용하는 정답표는 아닙니다.

## TODO 2 — 수집 후보 판단하기

`judge_source(item)`은 `(판단 문자열, 근거 문자열)` 두 값을 묶어서 반환합니다. 근거는 아래 문장을 그대로 복사할 필요 없이 어떤 조건 때문에 판단했는지 드러나면 됩니다.

먼저 `personal_data`가 참이면 질문에 필요 없는 개인정보를 이유로 `제외`합니다. 그다음 `terms_ok`와 `reuse_ok` 중 하나라도 확인된 참이 아니면 `주의`로 둡니다. 둘을 확인했더라도 웹 자동 수집인데 `robots_ok`가 확인된 참이 아니면 `주의`입니다. 여기까지 통과하면 제공된 가상 조건 안에서 `가능`으로 판단합니다.

`주의`도 당장 수집을 진행하는 상태는 아닙니다. 미확인 조건을 확인하거나 허용되는 다른 출처를 찾아야 합니다. API와 파일에는 웹 자동 수집에 사용하는 robots 조건을 적용하지 않습니다. Python의 `is True` 또는 `is not True`로 확인 여부를 비교할 수 있습니다.

## TODO 3 — 판단과 출처를 한 기록에 담기

`make_record(item)`에는 출처 이름·형식·주소·이용 조건이 담긴 `record`가 제공됩니다. 여기에 다음 정보를 추가한 뒤 딕셔너리를 반환하세요.

- `route`: TODO 1 함수의 반환값
- `status`, `reason`: TODO 2가 반환한 두 값
- `checked_at`: 제공된 `CHECKED_AT`. 가상 이용 조건을 확인했다고 설정한 날짜입니다.
- `collected_at`: 아직 수집하지 않았으므로 `None`
- `ready`: 판단이 `가능`일 때만 참인 불리언

문자열 두 개를 반환한 함수의 결과는 변수 두 개에 나누어 받을 수 있습니다. `ready`에 `가능`이라는 문자열 자체를 저장하지 않도록 주의하세요.

## 실행하기

시작 코드의 `raise NotImplementedError(...)` 세 곳을 구현한 뒤 위에서부터 실행하세요. 함수 정의만 실행하면 파일이 생기지 않습니다. 마지막 셀의 `make_record` 호출과 저장 코드까지 실행해야 결과를 볼 수 있습니다.

노트북에서는 현재 작업 폴더에, Python 파일로 실행하면 그 파일 옆에 `collection_plan.json`이 생깁니다. 같은 위치에서 다시 실행하면 이 계획 파일을 갱신합니다. 저장 코드는 제공되어 있으므로 경로를 판단하는 세 함수에 집중하면 됩니다.

## 실행 준비

**Windows + VS Code + PowerShell + Python 3.12 + uv**

아래 자료 ZIP을 내려받아 압축을 풀고, `pyproject.toml`이 있는 폴더를 VS Code로 엽니다. Python과 Jupyter 확장을 설치하고 **터미널 → 새 터미널**에서 PowerShell을 선택하세요. uv가 없다면 먼저 `python -m pip install uv`를 실행합니다.

```powershell
uv sync --python 3.12
uv run python --version
uv run python starter.py
```

버전 출력이 `Python 3.12.x`인지 확인합니다. 제공된 `.python-version`과 `pyproject.toml`도 Python 3.12를 지정합니다. 해당 Python이 없으면 uv가 준비합니다. 별도의 가상환경 활성화 명령은 필요하지 않습니다.

노트북으로 풀려면 `data_api_chapter01_starter.ipynb`를 열고 오른쪽 위 **커널 선택**에서 이 폴더의 `.venv`를 선택합니다. `uv sync`를 마쳤다면 노트북의 패키지 설치 셀은 건너뛰고 준비 코드부터 실행하세요.

**선택: Google Colab**

Colab을 사용할 때는 **파일 → 노트북 업로드**에서 `data_api_chapter01_starter.ipynb`를 엽니다. 필요한 데이터는 노트북에 포함되어 있습니다. 별도 패키지 설치 없이 실행합니다.

시작 코드의 세 TODO는 작성 전 `NotImplementedError`로 멈추도록 되어 있습니다. 각 묶음을 채운 뒤 처음부터 다시 실행하세요.

## 1. 사내 도우미가 답할 질문과 가상 출처

In [1]:
import json
from pathlib import Path

# 아래 조건과 주소는 연습을 위해 만든 자료이며 실제 이용 허가가 아닙니다.
CHECKED_AT = "2026-09-08"
CANDIDATES = [
    {"id": "policy", "name": "교육비 지원 규정", "format": "PDF",
     "question": "외부 교육비는 어떤 조건에서 지원되나요?",
     "needs_history": False, "changes_often": False, "needs_explanation": True,
     "method": "file", "personal_data": False, "terms_ok": True,
     "reuse_ok": True, "robots_ok": None,
     "condition": "가상 사내 규정: 출처를 표시하면 내부 검색용 저장 허용"},
    {"id": "seats", "name": "교육 신청 잔여석", "format": "JSON",
     "question": "지금 신청할 수 있는 자리는 몇 개인가요?",
     "needs_history": False, "changes_often": True, "needs_explanation": False,
     "method": "api", "personal_data": False, "terms_ok": True,
     "reuse_ok": True, "robots_ok": None,
     "condition": "가상 API 조건: 내부 조회 허용, 응답 기준 시각 표시"},
    {"id": "trend", "name": "월별 교육 신청 통계", "format": "HTML",
     "question": "최근 석 달의 교육 신청 수는 어떻게 달라졌나요?",
     "needs_history": True, "changes_often": True, "needs_explanation": False,
     "method": "web", "personal_data": False, "terms_ok": True,
     "reuse_ok": True, "robots_ok": None,
     "condition": "가상 이용 조건: 출처 표시 후 보관 허용, robots는 미확인"},
    {"id": "contacts", "name": "교육 신청자 연락처", "format": "CSV",
     "question": "교육 담당 부서의 이름을 알려 주세요.",
     "needs_history": False, "changes_often": False, "needs_explanation": False,
     "method": "file", "personal_data": True, "terms_ok": True,
     "reuse_ok": None, "robots_ok": None,
     "condition": "질문에 필요 없는 개인 전화번호 포함, 재사용 조건 미확인"},
]



## 2. 경로를 고르고 이용 조건 판단하기

In [ ]:
def choose_route(item):
    # TODO 1: 과거 비교, 현재값, 설명, 그 밖의 질문 순으로 판단합니다.
    raise NotImplementedError("TODO를 구현하세요.")
    # TODO 1 끝


def judge_source(item):
    # TODO 2: 모르는 조건(None)은 확인한 조건(True)으로 취급하지 않습니다.
    raise NotImplementedError("TODO를 구현하세요.")
    # TODO 2 끝


def make_record(item):
    # 주소와 원문 조건은 제공하고, 판단 결과와 시각은 직접 기록합니다.
    record = {"name": item["name"], "format": item["format"],
              "source_url": f"https://training.example/{item['id']}",
              "terms_url": f"https://training.example/{item['id']}/terms",
              "condition": item["condition"]}
    # TODO 3: 수집 계획을 만드는 단계이므로 collected_at은 아직 없습니다.
    raise NotImplementedError("TODO를 구현하세요.")
    # TODO 3 끝




## 3. 수집 계획 확인하고 저장하기

In [ ]:
records = [make_record(item) for item in CANDIDATES]
for record in records:
    print(record["name"], "|", record["route"], "|", record["status"])
    print("  근거:", record["reason"])
print("진행 후보:", [row["name"] for row in records if row["ready"]])
# 스크립트는 파일 옆에, 노트북은 현재 폴더에 결과를 저장합니다.
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
output = base / "collection_plan.json"
output.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
print("계획 파일:", output)


In [1]:
# %% 1. 사내 도우미가 답할 질문과 가상 출처
import json
from pathlib import Path

# 아래 조건과 주소는 연습을 위해 만든 자료이며 실제 이용 허가가 아닙니다.
CHECKED_AT = "2026-09-08"
CANDIDATES = [
    {"id": "policy", "name": "교육비 지원 규정", "format": "PDF",
     "question": "외부 교육비는 어떤 조건에서 지원되나요?",
     "needs_history": False, "changes_often": False, "needs_explanation": True,
     "method": "file", "personal_data": False, "terms_ok": True,
     "reuse_ok": True, "robots_ok": None,
     "condition": "가상 사내 규정: 출처를 표시하면 내부 검색용 저장 허용"},
    {"id": "seats", "name": "교육 신청 잔여석", "format": "JSON",
     "question": "지금 신청할 수 있는 자리는 몇 개인가요?",
     "needs_history": False, "changes_often": True, "needs_explanation": False,
     "method": "api", "personal_data": False, "terms_ok": True,
     "reuse_ok": True, "robots_ok": None,
     "condition": "가상 API 조건: 내부 조회 허용, 응답 기준 시각 표시"},
    {"id": "trend", "name": "월별 교육 신청 통계", "format": "HTML",
     "question": "최근 석 달의 교육 신청 수는 어떻게 달라졌나요?",
     "needs_history": True, "changes_often": True, "needs_explanation": False,
     "method": "web", "personal_data": False, "terms_ok": True,
     "reuse_ok": True, "robots_ok": None,
     "condition": "가상 이용 조건: 출처 표시 후 보관 허용, robots는 미확인"},
    {"id": "contacts", "name": "교육 신청자 연락처", "format": "CSV",
     "question": "교육 담당 부서의 이름을 알려 주세요.",
     "needs_history": False, "changes_often": False, "needs_explanation": False,
     "method": "file", "personal_data": True, "terms_ok": True,
     "reuse_ok": None, "robots_ok": None,
     "condition": "질문에 필요 없는 개인 전화번호 포함, 재사용 조건 미확인"},
]

# %% 2. 경로를 고르고 이용 조건 판단하기
def choose_route(item):
    # TODO 1: 과거 비교, 현재값, 설명, 그 밖의 질문 순으로 판단합니다.
    # 월별 변화는 현재 응답 하나로 복원할 수 없어 갱신 빈도보다 과거 보관을 먼저 봅니다.
    if item["needs_history"]:
        return "주기적 파일/스냅샷"
    if item["changes_often"]:
        return "질문 시점 API"
    if item["needs_explanation"]:
        return "공식 문서/RAG"
    return "정적 파일"
    # TODO 1 끝


def judge_source(item):
    # TODO 2: 모르는 조건(None)은 확인한 조건(True)으로 취급하지 않습니다.
    # 이 질문에 불필요한 개인정보라면 다른 조건을 더 확인하기 전에 제외합니다.
    if item["personal_data"]:
        return "제외", "질문에 필요 없는 개인정보"
    if item["terms_ok"] is not True or item["reuse_ok"] is not True:
        return "주의", "이용 또는 재사용 조건 확인 필요"
    if item["method"] == "web" and item["robots_ok"] is not True:
        return "주의", "웹 수집 경로의 robots 확인 필요"
    return "가능", "제공된 가상 조건 안에서 사용 가능"
    # TODO 2 끝


def make_record(item):
    # 주소와 원문 조건은 제공하고, 판단 결과와 시각은 직접 기록합니다.
    record = {"name": item["name"], "format": item["format"],
              "source_url": f"https://training.example/{item['id']}",
              "terms_url": f"https://training.example/{item['id']}/terms",
              "condition": item["condition"]}
    # TODO 3: 수집 계획을 만드는 단계이므로 collected_at은 아직 없습니다.
    record["route"] = choose_route(item)
    record["status"], record["reason"] = judge_source(item)
    record["checked_at"] = CHECKED_AT
    record["collected_at"] = None
    # 판단 기록은 네 건 모두 보관하고, 후속 수집 대상만 이 불리언으로 고릅니다.
    record["ready"] = record["status"] == "가능"
    return record
    # TODO 3 끝


# %% 3. 수집 계획 확인하고 저장하기
records = [make_record(item) for item in CANDIDATES]
for record in records:
    print(record["name"], "|", record["route"], "|", record["status"])
    print("  근거:", record["reason"])
print("진행 후보:", [row["name"] for row in records if row["ready"]])
# 스크립트는 파일 옆에, 노트북은 현재 폴더에 결과를 저장합니다.
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
output = base / "collection_plan.json"
output.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
print("계획 파일:", output)

교육비 지원 규정 | 공식 문서/RAG | 가능
  근거: 제공된 가상 조건 안에서 사용 가능
교육 신청 잔여석 | 질문 시점 API | 가능
  근거: 제공된 가상 조건 안에서 사용 가능
월별 교육 신청 통계 | 주기적 파일/스냅샷 | 주의
  근거: 웹 수집 경로의 robots 확인 필요
교육 신청자 연락처 | 정적 파일 | 제외
  근거: 질문에 필요 없는 개인정보
진행 후보: ['교육비 지원 규정', '교육 신청 잔여석']
계획 파일: /content/collection_plan.json


## 결과 확인

계획 파일에는 후보 네 건이 모두 있어야 합니다. 판단이 끝난 뒤 `가능`한 것만 남기면 보류하거나 제외한 이유를 나중에 확인할 수 없으므로, 전체 기록을 보관합니다.

| 후보 | 수집 경로 | 판단 |
| --- | --- | --- |
| 교육비 지원 규정 | 공식 문서/RAG | 가능 |
| 교육 신청 잔여석 | 질문 시점 API | 가능 |
| 월별 교육 신청 통계 | 주기적 파일/스냅샷 | 주의 |
| 교육 신청자 연락처 | 정적 파일 | 제외 |

출력의 진행 후보는 앞의 두 자료입니다. 네 기록에 출처 주소, 이용 조건, 확인일이 남아 있고 `collected_at`은 모두 JSON의 `null`인지 확인하세요. `ready`는 문자열이 아닌 `true` 또는 `false`여야 합니다.

## 짧은 관찰 메모

1. 월별 통계는 자주 바뀌는데도 왜 `질문 시점 API`보다 스냅샷을 먼저 골랐나요? 현재 응답만 보관했을 때 답할 수 없는 질문을 하나 적으세요.
2. 잔여석 API와 통계 웹페이지는 `robots_ok`가 둘 다 `None`입니다. 두 자료의 판단이 다른 이유와, 통계의 robots만 확인해도 재사용 조건이 불명확하다면 어떻게 판단할지 적으세요.
3. 자료가 무료여도 남는 작업 비용을 두 가지 적으세요. 그리고 이 결과의 확인일을 실제 수집 시각으로 사용하면 어떤 기록이 잘못되는지 설명하세요.

이 장은 약 45분 동안 TODO 구현과 결과 확인, 관찰 메모 작성까지 진행합니다. 완성한 코드 또는 노트북과 `collection_plan.json`을 제출하세요. 네 후보를 모두 남기고 진행 후보는 두 건인지 확인한 뒤, 월별 통계가 스냅샷으로 분류된 이유와 보류 조건을 관찰 메모에 적습니다.

## 나의 관찰 메모

위의 관찰 질문에 대한 답을 이 텍스트 셀에 작성하세요.
